# Optional LLM integration

## Goal

This notebook shows provider selection, environment-based configuration, and the privacy-safe default that excludes raw samples from prompts. It performs no network call unless `AUTOPREPML_RUN_LLM_EXAMPLE=1` is explicitly set.

In [1]:
import os
import pandas as pd
from autoprepml import LLMProvider, LLMSuggestor

print("supported_providers:", [provider.value for provider in LLMProvider])
sample = pd.DataFrame({"customer_age": [25, 31, None], "plan": ["basic", "pro", "basic"]})
requested_provider = os.getenv("AUTOPREPML_EXAMPLE_LLM_PROVIDER", "ollama")
run_network_call = os.getenv("AUTOPREPML_RUN_LLM_EXAMPLE", "0") == "1"

supported_providers: ['openai', 'anthropic', 'google', 'ollama']


### Initialize safely

In [2]:
suggestor = None
try:
    suggestor = LLMSuggestor(provider=requested_provider, include_samples=False)
    print("provider:", suggestor.provider.value)
    print("model:", suggestor.model)
    print("include_samples:", suggestor.include_samples)
except (ImportError, ValueError, RuntimeError) as error:
    print("provider_unavailable:", type(error).__name__)
    print("install the optional llm extra and configure the provider before making a call")

provider: ollama
model: llama3.2
include_samples: False


### Make an explicit, optional request

Set the provider API key or start the local Ollama service outside this notebook. The example intentionally does not print credentials or send data by default.

In [3]:
if run_network_call and suggestor is not None:
    response = suggestor.suggest_fix(sample, column="customer_age", issue_type="missing")
    print("response_preview:", str(response)[:300])
else:
    print("network_call_skipped: set AUTOPREPML_RUN_LLM_EXAMPLE=1 to opt in")

network_call_skipped: set AUTOPREPML_RUN_LLM_EXAMPLE=1 to opt in


## Checks

In [4]:
if suggestor is not None:
    assert suggestor.include_samples is False
assert sample["customer_age"].isna().sum() == 1
print("LLM integration configuration checks passed.")

LLM integration configuration checks passed.
